In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
# denote the file paths for the datasets to be used in the data analysis
traffic_file_path = r'data_files\Automated_Traffic_Volume_Counts_20260418.csv'
fine_particles_file_path = r'data_files\NYC EH Data Portal - Fine particles (PM 2.5) (full table).csv'
nitrogen_dioxide_file_path = r'data_files\NYC EH Data Portal - Nitrogen dioxide (NO2) (full table).csv'

In [3]:
# read in the datasets as pandas dataframes
traffic_df = pd.read_csv(traffic_file_path)
fine_particles_df = pd.read_csv(fine_particles_file_path)
nitrogen_dioxide_df = pd.read_csv(nitrogen_dioxide_file_path)

In [4]:
# exploratory data analyis for traffic dataset to check for missing values and data types
print(f'Traffic Columns: {traffic_df.columns}\n')
print(f'Traffic Shape: {traffic_df.shape}\n')
print(f'Traffic DF Head:\n{traffic_df.head()}\n')
print('Traffic DF Info:')
traffic_df.info()

Traffic Columns: Index(['RequestID', 'Boro', 'Yr', 'M', 'D', 'HH', 'MM', 'Vol', 'SegmentID',
       'WktGeom', 'street', 'fromSt', 'toSt', 'Direction'],
      dtype='str')

Traffic Shape: (1875154, 14)

Traffic DF Head:
   RequestID    Boro    Yr  M  D  HH  MM Vol  SegmentID  \
0      12512  Queens  2013  3  7   4  15   5      55135   
1      12512  Queens  2013  3  7   4  30   8      55135   
2      12512  Queens  2013  3  7   4  45   8      55135   
3      12512  Queens  2013  3  7   5   0   7      55135   
4      12512  Queens  2013  3  7   5  15   9      55135   

                      WktGeom  street     fromSt           toSt Direction  
0  POINT (1035363.4 185093.4)  122 PL  SUTTER AV  ROCKAWAY BLVD        SB  
1  POINT (1035363.4 185093.4)  122 PL  SUTTER AV  ROCKAWAY BLVD        SB  
2  POINT (1035363.4 185093.4)  122 PL  SUTTER AV  ROCKAWAY BLVD        SB  
3  POINT (1035363.4 185093.4)  122 PL  SUTTER AV  ROCKAWAY BLVD        SB  
4  POINT (1035363.4 185093.4)  122 PL  SUTTER

In [5]:
for col in traffic_df.columns:
    print(f'Column {col}: {traffic_df[col].dtype}\n')
    print(f'Traffic DF Unique Values for {col}:{traffic_df[col].unique()}\n')

Column RequestID: int64

Traffic DF Unique Values for RequestID:[12512 27853 28842 ...  9568 21450 21077]

Column Boro: str

Traffic DF Unique Values for Boro:<StringArray>
['Queens', 'Bronx', 'Brooklyn', 'Manhattan', 'Staten Island']
Length: 5, dtype: str

Column Yr: int64

Traffic DF Unique Values for Yr:[2013 2018 2009 2023 2012 2016 2008 2014 2021 2010 2011 2015 2017 2019
 2024 2022 2020 2007 2006 2000 2026 2025]

Column M: int64

Traffic DF Unique Values for M:[ 3  4 11 10  5  9  7  1 12  2  6  8]

Column D: int64

Traffic DF Unique Values for D:[ 7  8 10  9 18  5  3  4 15 16 30 31  1 12 13 24 25 21 22 14 29 19 26 17
 11  2 27 28  6 23 20]

Column HH: int64

Traffic DF Unique Values for HH:[ 4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23  0  1  2  3]

Column MM: int64

Traffic DF Unique Values for MM:[15 30 45  0 40 50 10 20]

Column Vol: str

Traffic DF Unique Values for Vol:<StringArray>
[    '5',     '8',     '7',     '9',     '4',    '13',    '27',    '17',
    '26

In [6]:
#drop columns that are not needed for the analysis
# street, fromSt, and toSt are not needed for the analysis as the analysis is conducted by borough
# SegmentID and WktGeom were retained as they may be useful for future geospatial analysis, compared to the street data
traffic_cols_to_remove = ['street', 'fromSt', 'toSt']
traffic_df_cleaned = traffic_df.drop(columns=traffic_cols_to_remove)
traffic_df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 1875154 entries, 0 to 1875153
Data columns (total 11 columns):
 #   Column     Dtype
---  ------     -----
 0   RequestID  int64
 1   Boro       str  
 2   Yr         int64
 3   M          int64
 4   D          int64
 5   HH         int64
 6   MM         int64
 7   Vol        str  
 8   SegmentID  int64
 9   WktGeom    str  
 10  Direction  str  
dtypes: int64(7), str(4)
memory usage: 157.4 MB


In [7]:
# find any rows with null values in the traffic dataset
print(f'Percentage of Null Values:\n{traffic_df_cleaned.isnull().sum() / len(traffic_df_cleaned) * 100}\n')

Percentage of Null Values:
RequestID    0.0
Boro         0.0
Yr           0.0
M            0.0
D            0.0
HH           0.0
MM           0.0
Vol          0.0
SegmentID    0.0
WktGeom      0.0
Direction    0.0
dtype: float64



In [8]:
#drop rows with Y column values outside analysis range of 2009 - 2025
traffic_df_cleaned = traffic_df_cleaned[(traffic_df_cleaned['Yr'] >= 2009) & (traffic_df_cleaned['Yr'] <= 2025)]

#check the M, D, HH, and MM columns for outliers
traffic_df_cleaned = traffic_df_cleaned[(traffic_df_cleaned['M'] >= 1) & (traffic_df_cleaned['M'] <= 12)]
traffic_df_cleaned = traffic_df_cleaned[(traffic_df_cleaned['D'] >= 1) & (traffic_df_cleaned['D'] <= 31)]
traffic_df_cleaned = traffic_df_cleaned[(traffic_df_cleaned['HH'] >= 0) & (traffic_df_cleaned['HH'] <= 23)]
traffic_df_cleaned = traffic_df_cleaned[(traffic_df_cleaned['MM'] >= 0) & (traffic_df_cleaned['MM'] <= 59)]

#convert the Y, M, D, HH, and MM individual columns into a single column with datetime format
traffic_df_cleaned['DateTime'] = pd.to_datetime({
     'year': traffic_df_cleaned['Yr'],
     'month': traffic_df_cleaned['M'],
     'day': traffic_df_cleaned['D'],
     'hour': traffic_df_cleaned['HH'],
     'minute': traffic_df_cleaned['MM']}, errors='coerce')

#drop the original Y, M, D, HH, and MM columns as they are no longer needed after creating the date_time column
traffic_df_cleaned = traffic_df_cleaned.drop(columns=['Yr', 'M', 'D','HH', 'MM'])
traffic_df_cleaned.info()



<class 'pandas.DataFrame'>
Index: 1822854 entries, 0 to 1875153
Data columns (total 7 columns):
 #   Column     Dtype         
---  ------     -----         
 0   RequestID  int64         
 1   Boro       str           
 2   Vol        str           
 3   SegmentID  int64         
 4   WktGeom    str           
 5   Direction  str           
 6   DateTime   datetime64[us]
dtypes: datetime64[us](1), int64(2), str(4)
memory usage: 111.3 MB


In [9]:
#convert Vol column to numeric, coercing errors to NaN
#fix the formatting in the Vol column by removing commas and converting to numeric
traffic_df_cleaned['Vol'] = traffic_df_cleaned['Vol'].str.replace(',', '')
traffic_df_cleaned['Vol'] = pd.to_numeric(traffic_df_cleaned['Vol'], errors = 'coerce')
#check for null values df
print(f'Null Value Percentage Check:\n{traffic_df_cleaned.isnull().sum() / len(traffic_df_cleaned) * 100}\n')
traffic_df_cleaned.info()

Null Value Percentage Check:
RequestID    0.0
Boro         0.0
Vol          0.0
SegmentID    0.0
WktGeom      0.0
Direction    0.0
DateTime     0.0
dtype: float64

<class 'pandas.DataFrame'>
Index: 1822854 entries, 0 to 1875153
Data columns (total 7 columns):
 #   Column     Dtype         
---  ------     -----         
 0   RequestID  int64         
 1   Boro       str           
 2   Vol        int64         
 3   SegmentID  int64         
 4   WktGeom    str           
 5   Direction  str           
 6   DateTime   datetime64[us]
dtypes: datetime64[us](1), int64(3), str(3)
memory usage: 111.3 MB


In [10]:
#check for duplicated rows in the traffic dataset
print(f'Number of Duplicated Rows in Traffic DF: {traffic_df_cleaned.duplicated().sum()}\n')

#drop duplicated rows if there are any
traffic_df_cleaned = traffic_df_cleaned.drop_duplicates()
print(f'Number of Duplicated Rows in Traffic DF After Dropping Duplicates: {traffic_df_cleaned.duplicated().sum()}\n')

Number of Duplicated Rows in Traffic DF: 0

Number of Duplicated Rows in Traffic DF After Dropping Duplicates: 0



In [11]:
#check for negative values in the Vol column
print(f'Number of Records in Traffic Dataset: {len(traffic_df_cleaned)}\n')
print(f'Number of Records with Negative Vol Values: {len(traffic_df_cleaned[traffic_df_cleaned["Vol"] < 0])}\n')
traffic_df_cleaned = traffic_df_cleaned[traffic_df_cleaned['Vol']>=0]
print(f'Number of Records with Negative Vol Values After Dropping Negative Values: {len(traffic_df_cleaned[traffic_df_cleaned["Vol"] < 0])}\n')
print(f'Number of Records in Traffic Dataset After Dropping Negative Vol Values: {len(traffic_df_cleaned)}\n')

#check for outliers in Vol column
q1_vol = traffic_df_cleaned['Vol'].quantile(0.25)
q3_vol = traffic_df_cleaned['Vol'].quantile(0.75)
iqr_vol = q3_vol - q1_vol
lower_bound_vol = q1_vol - (1.5 * iqr_vol)
upper_bound_vol = q3_vol + (1.5 * iqr_vol)
outliers = traffic_df_cleaned[(traffic_df_cleaned['Vol'] < lower_bound_vol) | (traffic_df_cleaned['Vol'] > upper_bound_vol)]
print(f'Number of Records in Traffic Dataset: {len(traffic_df_cleaned)}')
print(f'Number of Outliers in Vol Column: {len(outliers)}\n')
print(f'Outliers Shape: {outliers.shape}\n')
print(f'Outliers Head:\n{outliers.head()}\n')
print(f'Outliers Description:\n{outliers["Vol"].describe()}\n')



Number of Records in Traffic Dataset: 1822854

Number of Records with Negative Vol Values: 1

Number of Records with Negative Vol Values After Dropping Negative Values: 0

Number of Records in Traffic Dataset After Dropping Negative Vol Values: 1822853

Number of Records in Traffic Dataset: 1822853
Number of Outliers in Vol Column: 126062

Outliers Shape: (126062, 7)

Outliers Head:
      RequestID       Boro  Vol  SegmentID  \
1575      36828  Manhattan  549      36369   
1576      36828  Manhattan  501      36369   
1577      36828  Manhattan  524      36369   
1578      36828  Manhattan  569      36369   
1579      36828  Manhattan  516      36369   

                                           WktGeom Direction  \
1575  POINT (994214.9732317923 216043.31446641852)        NB   
1576  POINT (994214.9732317923 216043.31446641852)        NB   
1577  POINT (994214.9732317923 216043.31446641852)        NB   
1578  POINT (994214.9732317923 216043.31446641852)        NB   
1579  POINT (9942

In [12]:
#add a Vol_log column to the traffic dataset to use in the analysis to mitigate the effect of outliers in the Vol column without remoing the data
traffic_df_cleaned['Vol_log'] = np.log1p(traffic_df_cleaned['Vol'] + 1)
print(f'Updated Traffic Shape: {traffic_df_cleaned.shape}\n')
print(f'Traffic Head:\n{traffic_df_cleaned.head(10)}\n')
print(f'Traffic Description:\n{traffic_df_cleaned["Vol_log"].describe()}\n')

Updated Traffic Shape: (1822853, 8)

Traffic Head:
   RequestID    Boro  Vol  SegmentID                     WktGeom Direction  \
0      12512  Queens    5      55135  POINT (1035363.4 185093.4)        SB   
1      12512  Queens    8      55135  POINT (1035363.4 185093.4)        SB   
2      12512  Queens    8      55135  POINT (1035363.4 185093.4)        SB   
3      12512  Queens    7      55135  POINT (1035363.4 185093.4)        SB   
4      12512  Queens    9      55135  POINT (1035363.4 185093.4)        SB   
5      12512  Queens    4      55135  POINT (1035363.4 185093.4)        SB   
6      12512  Queens    8      55135  POINT (1035363.4 185093.4)        SB   
7      12512  Queens   13      55135  POINT (1035363.4 185093.4)        SB   
8      12512  Queens   27      55135  POINT (1035363.4 185093.4)        SB   
9      12512  Queens   17      55135  POINT (1035363.4 185093.4)        SB   

             DateTime   Vol_log  
0 2013-03-07 04:15:00  1.945910  
1 2013-03-07 04:30:00 

In [13]:
#identify outliers in the Vol_log column
q1_vol_log = traffic_df_cleaned['Vol_log'].quantile(0.25)
q3_vol_log = traffic_df_cleaned['Vol_log'].quantile(0.75)
iqr_vol_log = q3_vol_log - q1_vol_log
lower_bound_vol_log = q1_vol_log - (1.5 * iqr_vol_log)
upper_bound_vol_log = q3_vol_log + (1.5 * iqr_vol_log)
outliers_vol_log = traffic_df_cleaned[(traffic_df_cleaned['Vol_log'] < lower_bound_vol_log) | (traffic_df_cleaned['Vol_log'] > upper_bound_vol_log)]
print(f'Number of Outliers in Vol_log Column: {len(outliers_vol_log)}\n')
print(f'Outliers Vol_log Shape: {outliers_vol_log.shape}\n')
print(f'Outliers Description:\n{outliers_vol_log["Vol_log"].describe()}\n')

Number of Outliers in Vol_log Column: 0

Outliers Vol_log Shape: (0, 8)

Outliers Description:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: Vol_log, dtype: float64



In [14]:
#encode potential categorical variables in the traffic dataset as needed for future analysis
traffic_df_cleaned = pd.get_dummies(traffic_df_cleaned, columns=['Boro', 'Direction'])
traffic_df_cleaned.head()

,RequestID,Vol,SegmentID,WktGeom,DateTime,Vol_log,Boro_Bronx,Boro_Brooklyn,Boro_Manhattan,Boro_Queens,Boro_Staten Island,Direction_EB,Direction_EW,Direction_NB,Direction_NS,Direction_SB,Direction_WB
0,12512,5,55135,POINT (1035363.4 185093.4),2013-03-07 04:15:00,1.945910,False,False,False,True,False,False,False,False,False,True,False
1,12512,8,55135,POINT (1035363.4 185093.4),2013-03-07 04:30:00,2.302585,False,False,False,True,False,False,False,False,False,True,False
2,12512,8,55135,POINT (1035363.4 185093.4),2013-03-07 04:45:00,2.302585,False,False,False,True,False,False,False,False,False,True,False
3,12512,7,55135,POINT (1035363.4 185093.4),2013-03-07 05:00:00,2.197225,False,False,False,True,False,False,False,False,False,True,False
4,12512,9,55135,POINT (1035363.4 185093.4),2013-03-07 05:15:00,2.397895,False,False,False,True,False,False,False,False,False,True,False


In [15]:
#save the cleaned traffic dataset as a new csv file for use in the data analysis
traffic_df_cleaned = traffic_df_cleaned.sort_values(by='DateTime', ascending=False).reset_index(drop=True)
traffic_df_cleaned.to_csv(r'data_files\cleaned_traffic_data.csv', index=False)